# 09. 타입 힌트 (typing)

파이썬은 런타임에 타입을 강제하지 않지만, 타입 힌트를 달아두면 IDE 자동완성/정적 분석(mypy)이 버그를 미리 잡아줍니다. Kotlin/Spring Boot처럼 정적 타입 언어에 익숙하다면 꽤 자연스럽게 느껴질 부분입니다.

## 기본 문법: 변수와 함수

In [1]:
age: int = 30
name: str = "Noni"
scores: list[int] = [90, 85, 70]  # Python 3.9+부터 list[int] 처럼 내장 타입 자체를 제네릭으로 사용 가능

def add(a: int, b: int) -> int:
    return a + b

print(add(2, 3))
print(scores)

# 힌트는 '강제'가 아니라 '문서 + 분석 도구용 정보'일 뿐
def sloppy(a: int) -> int:
    return a  # 타입이 달라도 런타임 에러는 안 남

print(sloppy("실제로는 문자열도 그냥 통과됨"))


5
[90, 85, 70]
실제로는 문자열도 그냥 통과됨


## typing 모듈: Optional, Union, Callable, Any

Python 3.10+부터는 `Union[int, str]` 대신 `int | str`, `Optional[int]` 대신 `int | None`로 더 간결하게 쓸 수 있습니다.

In [2]:
from typing import Optional, Union, Callable, Any

def find_user(user_id: int) -> Optional[dict]:  # dict | None 과 동일
    users = {1: {"name": "noni"}}
    return users.get(user_id)

def parse(value: Union[int, str]) -> str:  # int | str 과 동일 (3.10+)
    return str(value)

def apply(func: Callable[[int, int], int], a: int, b: int) -> int:
    return func(a, b)

print(find_user(1), find_user(99))
print(parse(42), parse("hello"))
print(apply(lambda x, y: x + y, 3, 4))

# Python 3.10+ 최신 문법 (버전 확인 후 실행됨)
def modern_style(value: int | None = None) -> int | str:
    return value if value is not None else "없음"

print(modern_style(), modern_style(10))


{'name': 'noni'} None
42 hello
7
없음 10


## dataclass + 타입 힌트

06번에서 본 `@dataclass`는 사실 타입 힌트를 **필수로 요구**합니다 — 힌트가 필드 선언 역할을 겸합니다.

In [3]:
from dataclasses import dataclass

@dataclass
class TrainConfig:
    lr: float
    batch_size: int
    model_name: str = "resnet18"
    tags: list[str] = None  # 실무에서는 field(default_factory=list) 권장 (04번 mutable default 함정과 연결)

cfg = TrainConfig(lr=0.001, batch_size=32)
print(cfg)


TrainConfig(lr=0.001, batch_size=32, model_name='resnet18', tags=None)


## TypedDict: dict에 타입을 달기

설정값을 그냥 `dict`로 넘기는 코드가 많은데, `TypedDict`를 쓰면 어떤 키가 있어야 하는지 명시할 수 있습니다.

In [4]:
from typing import TypedDict

class SensorMessage(TypedDict):
    sensor_id: str
    timestamp: float
    value: float

def handle(msg: SensorMessage) -> None:
    print(f"[{msg['sensor_id']}] t={msg['timestamp']} v={msg['value']}")

handle({"sensor_id": "lidar-1", "timestamp": 1234.5, "value": 3.2})


[lidar-1] t=1234.5 v=3.2


## mypy로 정적 검사 (실행은 안 함, 개념만)

타입 힌트는 런타임에 체크되지 않으므로, 실제로 '틀렸을 때 알려주는' 역할은 `mypy` 같은 정적 분석 도구가 합니다.

```bash
pip install mypy
mypy my_script.py
```

이 노트북 예제처럼 `def add(a: int, b: int) -> int`에 문자열을 넘기면 mypy가 실행 전에 타입 에러를 잡아줍니다 — 런타임에선 안 잡히는 버그를 미리 발견.

## 헷갈렸던 점 / 더 찾아볼 것

- 타입 힌트는 '강제'가 아니라 '약속'이라는 점 — 팀 코드에서는 mypy나 pyright를 CI에 넣어야 실제로 강제됨
- `Protocol`(구조적 타이핑, duck typing을 타입으로 표현)은 나중에 필요해지면 찾아볼 것